# Basic RAG (Without semantic search)

Common prep steps for all the code blocks in this notebook

In [11]:
from openai import OpenAI
from dotenv import load_dotenv

def get_recipe_from_sql_query(recipe): 
    return "1.8kg floury potatoes cut in half, or quarters if large, 5 tbsp olive oil, 2 tsp plain flour, 100g parmesan." \
           "Heat oven to 220C/fan 200C/gas 7. Place the potatoes into a pan of salted water, bring to the boil and simmer " \
           "for 2 mins exactly. Drain well and toss in a little of the oil to coat."

load_dotenv()

True

Basic rag, the content to augment the generation is retrieved from sql:

In [10]:
recipe = get_recipe_from_sql_query("parmesan_roasted_potatoes")

client = OpenAI()
response = client.responses.create(
    model="gpt-5-nano",
    instructions="you're a chef assistant, your goal is to create a bullet list of ingredients in the following" \
                 " format: - [quantity] - [ingredient]. You will be given a recipe retrieved from a database, use" \
                 " only ingredients found in the input to answer. If the input doesn't include any ingredients," \
                 " say so explicitly instead of trying to answer with ungrounded info.",
    input=recipe
)

print(response.output_text)
print(f"\nusage: {response.usage.total_tokens}")

- 1.8kg - floury potatoes
- 5 tbsp - olive oil
- 2 tsp - plain flour
- 100 g - parmesan

usage: 1339


Another basic rag:
- the user selects a recipe using a GUI and asks a question about it
- the system retrieves the recipe from sql
- the system builds the prompt by appending the the user's question to the retrieved recipe

This example is closer to a realistic use case because it contains both retrieved content and a user question

In [ ]:
user_selected_recipe = "parmesan_roasted_potatoes"
user_question = "can you tell me the ingredients needed for this recipe?" 
sql_recipe = get_recipe_from_sql_query(user_selected_recipe)

input = f"{sql_recipe} \n--- User question:\n {user_question}" 

client = OpenAI()
response = client.responses.create(
    model="gpt-5-nano",
    instructions="you're a chef assistant, your goal is to answer the user's question. " \
                 "You will be given a recipe retrieved from a database, " \
                 "use only the info included in the recipe to generate the answer. " \
                 "If the recipe doesn't contain enough info to answer, say so explicitly instead of trying to answer with ungrounded info.",
    input=input
)

print(response.output_text)
print(f"\nusage: {response.usage.total_tokens}")

Here are the ingredients listed in the recipe:

- 1.8 kg floury potatoes, halved (or quarters if large)
- 5 tbsp olive oil
- 2 tsp plain flour
- 100 g parmesan
- Water for boiling, salted (as per the method)

usage: 1471
